### THINGS MEMORABILITY ANALYSIS 

##### Main code from Github 

In [1]:
# import packages
import platform
import torch
import pandas as pd
import sklearn as sk
import os
import torch.nn as nn
import numpy as np
import scipy
import thingsvision.vision as vision 
from thingsvision.model_class import Model
from thingsvision import get_extractor
from thingsvision.utils.storing import save_features #, split_activations, merge_activations
import thingsvision.utils.storing as storing
from thingsvision.utils.data import ImageDataset, DataLoader 
from thingsvision.core.extraction import center_features 

In [2]:
has_gpu = torch.cuda.is_available()
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"

In [1]:
print(f"Python Platform: {platform.platform()}")
print(f"PyTorch Version: {torch.__version__}")
print()
print(f"Python {sys.version}")
print(f"Pandas {pd.__version__}")
print(f"Scikit-Learn {sk.__version__}")
print("NVIDIA/CUDA GPU is", "available" if has_gpu else "NOT AVAILABLE")
print("MPS (Apple Metal) is", "AVAILABLE" if has_mps else "NOT AVAILABLE")
print(f"Target device is {device}")

In [6]:
path = '/add/your/path/here/'
savepath = '/add/your/path/here/'
cnn_layers_extract = True

In [11]:
def extract_features(
    extractor: Any,
    module_name: str,
    image_path: str,
    out_path: str,
    batch_size: int,
    flatten_activations: bool,
    apply_center_crop: bool,
    class_names: Optional[List[str]]=None,
    file_names: Optional[List[str]]=None,
) -> Union[np.ndarray, torch.Tensor]:
    """Extract features for a single layer."""                                    
    dataset = ImageDataset(
        root=image_path,
        out_path=out_path,
        backend=extractor.get_backend(),
        transforms=extractor.get_transformations(apply_center_crop=apply_center_crop, resize_dim=256, crop_dim=224),
        class_names=class_names,
        file_names=file_names,
    )
    batches = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        backend=extractor.get_backend(),
        )
    features = extractor.extract_features(
        batches=batches,
        module_name=module_name,
        flatten_acts=flatten_activations,
        output_type="ndarray", # or "tensor" (only applicable to PyTorch models)
    )
    return features


def extract_all_layers(
    model_name: str,
    extractor: Any,
    image_path: str,
    out_path: str,
    batch_size: int,
    flatten_activations: bool,
    apply_center_crop: bool,
    layer: Any=nn.Linear,
    file_format: str = "npy",
    class_names: Optional[List[str]]=None,
    file_names: Optional[List[str]]=None,
) -> Dict[str, Union[np.ndarray, torch.Tensor]]:
    """Extract features for all selected layers and save them to disk."""
    features_per_layer = {}
    for l, (module_name, module) in enumerate(extractor.model.named_modules(), start=1):
        if instance(module, layer):
            # extract features for layer "module_name"
            features = extract_features(
                extractor=extractor,
                module_name=module_name,
                image_path=image_path,
                out_path=out_path,
                batch_size=batch_size,
                flatten_activations=flatten_activations,
                apply_center_crop=apply_center_crop,
                class_names=class_names,
                file_names=file_names,
            )
            # if you want, replace for simplicity with e.g., [f'conv_{l:02d}'], [f'fc_{l:02d}'], or [f'layer_{l:02d}']
            features_per_layer[f'{module_name}'] = features
            # save features to disk
            #save_features(features, out_path=f'{out_path}/features_{model_name}_{module_name}', file_format=file_format)
    return features_per_layer

In [13]:
pretrained = False # use pretrained model weights
model_path = None # if pretrained = False (i.e., randomly initialized weights) set path to model weights
batch_size = 16 # use a power of two (this can be any size, depending on the number of images for which you aim to extract features)
apply_center_crop = True # center crop images (set to False, if you don't want to center-crop images)
flatten_activations = True # whether or not features (e.g., of Conv layers) should be flattened
class_names = None  # optional list of class names for class dataset
file_names = None # optional list of file names according to which features should be sorted
file_format = "npy" # format with which to save features to disk (can be set to "mat", "txt", "npy", "hdf5")
#device = 'cuda' if torch.cuda.is_available() else 'cpu'
#device = 'cpu'

In [12]:
# load model
model_name = 'alexnet'#

# specify model source 
# we use torchvision here (https://pytorch.org/vision/stable/models.html)
source = 'torchvision'
# initialize the extractor
extractor = get_extractor(
    model_name=model_name,
    pretrained=pretrained,
    model_path=model_path,
    device=device,
    source=source
)

In [11]:
#vision.show_model(model, model_name)
extractor.show_model() 

In [18]:
things_images_list = ['object_imagesA-C', 'object_imagesD-K', 'object_imagesL-Q', 'object_imagesR-S', 'object_imagesT-Z']


In [3]:
## LOOP over images and features and save output 

things_images_list = ['object_imagesA-C', 'object_imagesD-K', 'object_imagesL-Q', 'object_imagesR-S', 'object_imagesT-Z']

if cnn_layers_extract:
# extract features from the ReLU layers 
    # mods_name = ['features.0', 'features.3', 'features.7'] 
    mods_name = ['features.11'] 
    #mods_name = ['classifier.5']

else: 
# extract features from the ReLU layers 
    mods_name = ['features.1', 'features.3', 'features.5', 
                 'features.7', 'features.10', 'features.12', 
                 'features.14', 'features.17', 'features.19',
                 'features.21','features.24', 'features.26',
                 'features.28',
                ]


for module_name in mods_name:
    #output_dir = savepath + '/THINGS_' + module_name + '_' + model_name 
    output_dir = savepath + '/THINGS_' + module_name + '_' + model_name + '_untrained'
    #print(output_dir)
    
    for things_images in things_images_list:
        image_dir = path + 'THINGS/' + things_images
        
        output_path = f'{output_dir}/features_{model_name}_{module_name}/features_{things_images}'
        #print(output_path)
    
        if os.path.exists(output_path):
            print("path exists continue to next")
        else: 
            # extract features for a single layer (e.g., penultimate)
            features = extract_features(
                extractor=extractor,
                module_name=module_name,
                image_path=image_dir,
                out_path=output_dir,
                batch_size=batch_size,
                flatten_activations=flatten_activations,
                apply_center_crop=apply_center_crop,
                class_names=class_names,
                file_names=file_names,
            )

            
            save_features(features, out_path=output_path, file_format=file_format)

### Compute the L2 norm of the THINGS dataset 

In [4]:
# output_dir
extractor.show_model() 

In [143]:
dataframes = []
module_name_extract = 'classifier.5'
#module_name_extract = 'features.11'
out_dir = '/Volumes/MemoProj/Memo_project/DATA/THINGS_'
for things_images in things_images_list:
    feature_dir = out_dir + module_name_extract + '_' + model_name + '/features_'+ model_name + '_' + module_name_extract + '/features_' + things_images

    feature_file = feature_dir + '/features.npy'
    ann_layers = np.load(feature_file)
    
    df = pd.DataFrame(ann_layers)
   
    # Append the DataFrame to the list
    dataframes.append(df)
 
# concatenate all the DataFrames into one big DataFrame
THINGS_features = pd.concat(dataframes, ignore_index=True)
 
#  rename the columns to 1..n
THINGS_features.columns = range(1, THINGS_features.shape[1] + 1)

In [145]:
def load_csv_file(filename):
    df = pd.read_csv(filename)
    return df

In [147]:
THINGS_image_name_file = '/add/your/path/here/'
THINGS_features_load = load_csv_file(THINGS_image_name_file)

In [148]:
THINGS_features_1sthalf = THINGS_features[:13000]
THINGS_features_2ndhalf = THINGS_features[13000:26107]
l2_norms_1st = np.linalg.norm(THINGS_features_1sthalf, axis=1)
l2_norms_2nd = np.linalg.norm(THINGS_features_2ndhalf, axis=1)

array1 = np.array(l2_norms_1st).reshape(-1,1)
array2 = np.array(l2_norms_2nd).reshape(-1,1)

concatenated_L2norms = np.vstack((array1, array2))
THINGS_features_load["L2_norm"] = concatenated_L2norms

#### Load THINGS memorability scores

In [151]:
filename = path + '/THINGS_Memorability_Scores.csv'

In [153]:
THINGS_memo = load_csv_file(filename)

In [154]:
THINGS_memo.head()

,image_nr,image_name,file_path,cr
0,1,aardvark_01b.jpg,images/aardvark/aardvark_01b.jpg,0.825000
1,2,aardvark_02s.jpg,images/aardvark/aardvark_02s.jpg,0.800000
2,3,aardvark_03s.jpg,images/aardvark/aardvark_03s.jpg,0.878049
3,4,aardvark_04s.jpg,images/aardvark/aardvark_04s.jpg,0.731707
4,5,aardvark_05s.jpg,images/aardvark/aardvark_05s.jpg,0.825000


In [5]:
print(THINGS_features_load['L2_norm'].isna().sum())
print(THINGS_memo['cr'].isna().sum())

In [6]:
THINGS_memo.isna().stack()[lambda x: x].index

In [159]:
# we found that index 9795 does not have a memorabiilty score.
THINGS_memo2 = THINGS_memo.drop(index=9795)
THINGS_features_load2 = THINGS_features_load.drop(index=9795)

In [9]:
corr_Thingsmemo_L2norm, p_value_Thingsmemo_L2norm = scipy.stats.spearmanr(THINGS_features_load2['L2_norm'], THINGS_memo2['cr'])
print("Pearson correlation coefficient recall:", corr_Thingsmemo_L2norm)
print("P-value recall:", p_value_Thingsmemo_L2norm)

In [161]:
#THINGS_memo2.to_csv('THINGS_memo_scores.csv', index =False)
THINGS_features_load2.to_csv('AlexNet_classifier5_L2norm.csv', index =False)